#The Attention Mechanism — Solving the Bottleneck

Yesterday (Day 91), we saw that the Encoder-Decoder model has a major flaw: it tries to cram an entire sentence into a single "Context Vector." This is like trying to summarize a whole book in one sentence—you lose the details.

Attention changed everything. Instead of looking at just one vector, the Decoder "looks back" at all the Encoder's hidden states and decides which specific words are important for the word it is currently translating.

1. #Bahdanau Attention vs. Luong Attention

There are two main types of Attention you should know:

1. **Bahdanau Attention (Additive):** The attention scores are calculated using a small hidden layer. It's often used in the original Seq2Seq papers.

2. **Luong Attention (Dot-Product):** A mathematically simpler and faster version that uses the dot product between vectors. This is the foundation for what eventually became the Transformer.

2. #Implementation (Custom Layer Logic)

In modern Keras, you can use the built-in Attention or AdditiveAttention layers. Here is how you integrate it into your Seq2Seq model from yesterday:

In [2]:
import tensorflow as tf
from tensorflow.keras.layers import Input, LSTM, Embedding, Dense, Attention, Concatenate
from tensorflow.keras.models import Model

# 1. Setup Inputs
encoder_inputs = Input(shape=(20,), name="encoder_inputs")
decoder_inputs = Input(shape=(20,), name="decoder_inputs")

# 2. Encoder
encoder_emb = Embedding(input_dim=5000, output_dim=128)(encoder_inputs)
# return_sequences=True is CRITICAL for Attention
encoder_outputs, state_h, state_c = LSTM(64, return_sequences=True, return_state=True)(encoder_emb)
encoder_states = [state_h, state_c]

# 3. Decoder
decoder_emb = Embedding(input_dim=5000, output_dim=128)(decoder_inputs)
decoder_lstm_output, _, _ = LSTM(64, return_sequences=True, return_state=True)(
    decoder_emb, initial_state=encoder_states
)

# 4. Attention Layer (The Fix)
# Query = Decoder hidden states, Value = Encoder hidden states
attention_layer = Attention(name="attention_layer")
attention_output = attention_layer([decoder_lstm_output, encoder_outputs])

# 5. Combine and Output
decoder_combined_context = Concatenate(axis=-1)([decoder_lstm_output, attention_output])
outputs = Dense(5000, activation='softmax')(decoder_combined_context)

# Define Model
model = Model([encoder_inputs, decoder_inputs], outputs)
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ encoder_inputs      │ (None, 20)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_inputs      │ (None, 20)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, 20, 128)   │    640,000 │ encoder_inputs[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_1         │ (None, 20, 128)   │    640,000 │ decoder_inputs[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm (LSTM)         │ [(None, 20, 64),  │     49,408 │ embedding[0][0]   │
│                     │ (None, 64),       │            │                   │
│                     │ (None, 64)]       │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_1 (LSTM)       │ [(None, 20, 64),  │     49,408 │ embedding_1[0][0… │
│                     │ (None, 64),       │            │ lstm[0][1],       │
│                     │ (None, 64)]       │            │ lstm[0][2]        │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ attention_layer     │ (None, 20, 64)    │          0 │ lstm_1[0][0],     │
│ (Attention)         │                   │            │ lstm[0][0]        │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 20, 128)   │          0 │ lstm_1[0][0],     │
│ (Concatenate)       │                   │            │ attention_layer[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 20, 5000)  │    645,000 │ concatenate[0][0] │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 2,023,816 (7.72 MB)

 Trainable params: 2,023,816 (7.72 MB)

 Non-trainable params: 0 (0.00 B)